# Begin

Idea (a-la distill process):
1) take agents from 17e - they know how to play Frostbite one way or another
2) gather pairs (screenshot, action) of they play
3) learn world model on these pairs

In [1]:
# @launchit.collected
# @launchit.collected_temp_config
# @launchit.collected_optuna
# @launchit.collected_initrd
# @launchit.collected_build_docker_launch

In [2]:
# @launchit.collected_manual_run_docker_launch
# @launchit.collected_optuna_run_docker_launch

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import socket
import copy
from collections import namedtuple, defaultdict, Counter, deque # @launchit.collect
import random
import math
import datetime
import json # @launchit.collect
import pprint # @launchit.collect
import re 
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import pickle # @launchit.collect
import IPython 
from enum import StrEnum, auto # @launchit.collect
import multiprocessing as mp
import queue

import lark # @launchit.collect

from tqdm.notebook import tqdm

import numpy as np # @launchit.collect
import cupy as cp
import einops
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as VF
import torch.optim
import torch.multiprocessing as torch_mp
from torch.nn.attention import SDPBackend, sdpa_kernel

import gymnasium as gym 
import ale_py

import optuna # @launchit.collect
from optuna.storages import JournalStorage # @launchit.collect
from optuna.storages.journal import JournalFileBackend # @launchit.collect
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
build_project_root_path = '${BUILD_PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
sys.path.append(os.path.join(build_project_root_path, 'lib')) # @launchit.collect
from cleanrl.cleanrl_utils.atari_wrappers import (  # isort:skip
    ClipRewardEnv,
    EpisodicLifeEnv,
    FireResetEnv,
    MaxAndSkipEnv,
    NoopResetEnv,
)
import lang_utils as lu # @launchit.collect
import array_utils as au # @launchit.collect
from math_utils import RecursiveAverageFilter, RecursiveMovingAverageFilter
from logging_utils import *
from artifact_registry import * # @launchit.collect
from torch_utils import *
import launchit 
from hp_utils import * # @launchit.collect
from metrics_collector import RmqSummaryWriter, S3SummaryWriter
from autoincrement import Autoincrement

# Init

In [4]:
# @launchit.collect
class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    DOCKER_LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

In [5]:
# @launchit.collect_temp_config
# @launchit.disable

# Construct temporary CONFIG object for build and bootstrap purposes
if '${LAUNCHIT_FNAME}' != '$' + '{LAUNCHIT_FNAME}':
    notebook_fname = '${LAUNCHIT_FNAME}'
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    subproject_name = os.path.basename(os.path.dirname(notebook_fname))
    
    if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
        # This (bootstrap) config is for the very initial phase of docker launch when one needs to load hyperparameters.
        # This config will be recreated soon to full-fledged Config
        CONFIG = namedtuple('BootstrapConfig', 'initrd_path, exec_mode')(
            initrd_path=os.path.join(project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK,
        )
        assert Logging._instance is None, 'Must create Logging instance with use_raw_stdout=True, but Logging instance is already created'
        # use_raw_stdout is necessary because sys.stdout is hijacked by papermill and this will lead to problems
        # with string duplications written to stdout by child process (worker)
        # see https://share.google/aimode/ffIcoqyaJdgMsEcoW
        Logging.get(use_raw_stdout=True)    
    else:
        # Config to build docker launch
        CONFIG = namedtuple('BuildConfig', 
                            'initrd_path, relative_initrd_path, relative_run_path, relative_metrics_suite_fname, ' + 
                            'model_group_uri, self_fname, relative_self_fname, self_name, subproject_name, ' + 
                            'docker_registry, exec_mode, ' +
                            'artifact_registry_cache, artifact_registry')(
            initrd_path=os.path.join(build_project_root_path, 'run', subproject_name, 'initrd-' + notebook_name),
            relative_initrd_path=os.path.join('run', subproject_name, 'initrd-' + notebook_name), # relative to build project root
            relative_run_path=os.path.join('run', subproject_name),
            relative_metrics_suite_fname=os.path.join('run', subproject_name, notebook_name + '.metrics_suite.json'),
            model_group_uri='${MODEL_GROUP_URI}',
            self_fname=notebook_fname,
            relative_self_fname=lu.when('/run/' in notebook_fname, 'run/', '') + os.path.join(subproject_name, notebook_basename),  # relative to build project root
            self_name=notebook_name,
            subproject_name=subproject_name,
            docker_registry='cr.selcloud.ru/neurolab',
            exec_mode=ExecMode.LAUNCH_NOTEBOOK,
            artifact_registry_cache={}, 
            artifact_registry=None,
        )
        CONFIG = CONFIG._replace(artifact_registry=ArtifactRegistry(CONFIG.model_group_uri, cache=CONFIG.artifact_registry_cache))
        os.makedirs(CONFIG.initrd_path, exist_ok=True)

    Logging.get()(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n')
# @launchit.stop

In [6]:
def create_config():
    config = namedtuple('Config', 
                        'host_name, ' +
                        'project_root_path, project_root_uri, model_group_uri, subproject_path, data_path, private_data_path, run_path, initrd_path, ' + 
                        'self_fname, self_name, metrics_suite_fname, ' +
                        'subproject_name,' +
                        'is_cuda, cuda_device, docker_registry, exec_mode, is_interactive')(
        host_name=socket.gethostname(),
        project_root_path=project_root_path,
        project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
        model_group_uri=None,
        subproject_path=os.path.abspath('.'),
        data_path=os.path.join(project_root_path, 'data'),
        private_data_path=None,
        run_path=None,
        initrd_path=None,
        self_fname=None,
        self_name=None,
        metrics_suite_fname=None,
        subproject_name=None,
        is_cuda=torch.cuda.is_available(),
        cuda_device='cpu',
        docker_registry='cr.selcloud.ru/neurolab',
        exec_mode=ExecMode.MASTER_NOTEBOOK,
        is_interactive=True,
    )

    env_cuda_device = os.environ.get('CUDA_DEVICE')
    
    if torch.cuda.is_available():
        default_cuda_device = 'cuda'
        config = config._replace(cuda_device=lu.coalesce(env_cuda_device, default_cuda_device))
    else:
        assert env_cuda_device is None, f'CUDA device "{env_cuda_device}" is requested but CUDA is NOT available!'
    
    if IPython.get_ipython() is None:
        module_fname = __file__
        module_basename = os.path.basename(module_fname)
        module_name, _ = os.path.splitext(module_basename)
        
        config = config._replace(self_fname=module_fname, self_name=module_name)
        config = config._replace(exec_mode=ExecMode.LAUNCH_MODULE)
    else:
        with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
            notebook_fname = json.load(cf).get('jupyter_session')

            if notebook_fname is None:
                notebook_fname = os.path.join(config.subproject_path, os.path.basename('${LAUNCHIT_FNAME}'))
                assert os.path.exists(notebook_fname)
            
            notebook_basename = os.path.basename(notebook_fname)
            notebook_name, notebook_ext = os.path.splitext(notebook_basename)
        
            m = re.match(r'(\w+)-Copy\d+$', notebook_name)
        
            if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook
    
            config = config._replace(self_fname=notebook_fname, self_name=notebook_name)
            
            is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None

            if is_launch:
                if os.path.exists(os.path.join(project_root_path, '.docker_launch')):
                    config = config._replace(exec_mode=ExecMode.DOCKER_LAUNCH_NOTEBOOK)
                else:
                    config = config._replace(exec_mode=ExecMode.LAUNCH_NOTEBOOK)
            else:
                assert config.exec_mode == ExecMode.MASTER_NOTEBOOK
    
    config = config._replace(is_interactive=config.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
    config = config._replace(subproject_name=os.path.basename(os.path.dirname(config.self_fname)))
    config = config._replace(model_group_uri=f'{config.project_root_uri}.{config.subproject_name}')
    config = config._replace(run_path=os.path.join(project_root_path, 'run', config.subproject_name))
    config = config._replace(initrd_path=os.path.join(project_root_path, 'run', config.subproject_name, 'initrd-' + config.self_name))
    config = config._replace(private_data_path=os.path.join(config.data_path, config.subproject_name))
    config = config._replace(metrics_suite_fname=os.path.join(config.run_path, config.self_name + '.metrics_suite.json'))
    return config

In [7]:
# @launchit.disable_worker
au.init()
LOG = Logging.get()
RNG = np.random.default_rng()
CONFIG = create_config()
LOG.app_name = CONFIG.self_name
LOG.enable('syslog', CONFIG.exec_mode == ExecMode.LAUNCH_MODULE)
LOG.enable('stdout', CONFIG.exec_mode in [ExecMode.MASTER_NOTEBOOK, ExecMode.LAUNCH_NOTEBOOK])
LOG.enable('verbose_stdout', CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK)
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)
LOG(f'{os.environ=}', when=not CONFIG.is_interactive)
os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)
os.makedirs(CONFIG.initrd_path, exist_ok=True)

CONFIG=
{'host_name': 'thinkbook',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'data_path': '/home/misha/dev/mine/neurolab/data',
 'private_data_path': '/home/misha/dev/mine/neurolab/data/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'initrd_path': '/home/misha/dev/mine/neurolab/run/18_rl/initrd-18a_world_model_01',
 'self_fname': '/home/misha/dev/mine/neurolab/18_rl/18a_world_model_01.ipynb',
 'self_name': '18a_world_model_01',
 'metrics_suite_fname': '/home/misha/dev/mine/neurolab/run/18_rl/18a_world_model_01.metrics_suite.json',
 'subproject_name': '18_rl',
 'is_cuda': False,
 'cuda_device': 'cpu',
 'docker_registry': 'cr.selcloud.ru/neurolab',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



# Hyperparameters

In [8]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN = auto()
    WORKER = auto()

LaunchComponent = namedtuple('LaunchComponent', 'name version uri main_asset_fname')
    
@dataclass(slots=True)
class Hyperparameters:
    # Launch
    launch_goal: LaunchGoal = lu.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED)
    launch_id: int = lu.from_str(int, '${MODEL_VERSION}', 0)

    @dataclass(slots=True)
    class System:
        comment: str = None
        random_seed: int = None
        is_torch_deterministic: bool = True
        is_torch_compile: bool = False
        use_amp: bool = True

    @dataclass(slots=True)
    class Model:
        d_model: int = 256
        layers_count: int = 3
        heads_count: int = 4
        ob_shape: tuple = (1, 84, 84)
        ob_grid_shape: tuple = (16, 16)
        sequence_length: int = 4
        attention_backend: str = 'MATH' # value of SDPBackend enum (getattr(SDPBackend, attention_backend))

    @dataclass(slots=True)
    class Rollout:
        steps_count: int = 512 
        envs_count: int = 32
        env_rams: list = None
        env_ram_patches: list = None
        env_stories: list = dataclasses.field(default_factory=lambda: ['*;*']) # story = RAM + RAM patch. '*' = any RAM/RAM patch
        
    @dataclass(slots=True)
    class Train:
        test_pack: str = None
        epochs_count: int = 1 
        batch_size: int = 32
        optimizer: str = 'AdamW'
        max_grad_norm: float = 1.0
        learn_rate: str = 'const(0.00025)'
        bce_loss_coef: str = 'const(1.0)'
        edge_loss_coef: str = 'const(1.0)'

    system: System = dataclasses.field(default_factory=System)
    agents: list = None 
    model: Model = dataclasses.field(default_factory=Model)
    rollout: Rollout = dataclasses.field(default_factory=Rollout)
    train: Train = dataclasses.field(default_factory=Train)

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        hp.system = Hyperparameters.System(**hp.system)
        hp.agents = hp.agents
        hp.model = Hyperparameters.Agent(**hp.model)
        hp.rollout = Hyperparameters.Env(**hp.rollout)
        hp.train = Hyperparameters.Video(**hp.train)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

    def launch_component(self):
        name = lu.when('${MODEL_NAME}' == '$' + '{MODEL_NAME}', CONFIG.self_name, '${MODEL_NAME}')
        return LaunchComponent(name=name, version=self.launch_id,  uri=f'{CONFIG.model_group_uri}.{name}', main_asset_fname=CONFIG.self_fname)

HP = Hyperparameters()

# Launch

## LaunchState

In [9]:
@dataclass(slots=True)
class LaunchState:
    mp_ctx: object = None
    optuna_trial: dict = None
    artifact_registry_cache: dict = None
    artifact_registry: object = None
    summary_writer: object = None
    agents: dict = None
    agent_params: object = None
    env_id: str = None
    env_is_episodic_life: bool = None
    world_model: object = None
    rollout_manager: object = None
    rollout_env_story_sampler: object = None

## Configure

In [10]:
# @launchit.disable
# @launchit.collect
HP.system.random_seed = 42
HP.system.is_torch_deterministic = True
HP.system.is_torch_compile = True
HP.system.use_amp = True
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True,
            'use_amp': True},
 'agents': None,
 'model': {'d_model': 256,
           'layers_count': 3,
           'heads_count': 4,
           'ob_shape': (1, 84, 84),
           'ob_grid_shape': (16, 16),
           'sequence_length': 4,
           'attention_backend': 'MATH'},
 'rollout': {'steps_count': 512,
             'envs_count': 32,
             'env_rams': None,
             'env_ram_patches': None,
             'env_stories': ['*;*']},
 'train': {'test_pack': None,
           'epochs_count': 1,
           'batch_size': 32,
           'optimizer': 'AdamW',
           'max_grad_norm': 1.0,
           'learn_rate': 'const(0.00025)',
           'bce_loss_coef': 'const(1.0)',
           'edge_loss_coef': 'const(1.0)'}}


## Create

In [11]:
# @launchit.disable_worker
assert type(CONFIG).__name__ == 'Config', 'LaunchState should be created for full-fledged Config instance only'
LS = LaunchState()
LS.mp_ctx = torch_mp.get_context('spawn') # Spawn is needed for CUDA, fork doesn't work within PyTorch

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.system.random_seed is not None:
    random.seed(HP.system.random_seed)
    torch.manual_seed(HP.system.random_seed)
    RNG = np.random.default_rng(HP.system.random_seed)    
    LOG(f'Random seed={HP.system.random_seed}')

if HP.system.is_torch_deterministic is not None:
    torch.backends.cudnn.deterministic = HP.system.is_torch_deterministic
    LOG(f'{torch.backends.cudnn.deterministic=}')

lc = HP.launch_component()

artifact_registry_type = os.environ.get('ARTIFACT_REGISTRY', '').upper()

if not artifact_registry_type:
    artifact_registry_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'NEXUS')

match artifact_registry_type:
    case 'NEXUS':
        kwargs = {}

        if os.environ.get('NEXUS_URL', None) is not None:
            kwargs['nexus_url'] = os.environ['NEXUS_URL']

        if os.environ.get('DOWNLOAD_NEXUS_URL', None) is not None:
            kwargs['download_nexus_url'] = os.environ['DOWNLOAD_NEXUS_URL']
        
        if CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
            assert os.path.exists(CONFIG.initrd_path)
            cache_fname = os.path.join(CONFIG.initrd_path, 'artifact_registry_cache.pkl')

            if os.path.exists(cache_fname):
                with open(cache_fname, 'rb') as f:
                    LS.artifact_registry_cache = picke.load(f)
                    assert isinstance(LS.artifact_registry_cache, dict)
                    LOG(f'Loaded artifact registry cache from "{cache_fname}"')
            
            kwargs['cache'] = LS.artifact_registry_cache
        
        LS.artifact_registry = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri, **kwargs)
        LOG(f'Created ArtifactRegistry')
    case 'S3':
        LS.artifact_registry = S3ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)
        LOG(f'Created S3ArtifactRegistry')
    case _:
        assert False, f'Unsupported {artifact_registry_type=}'

if lc.version != 0:
    assert CONFIG.exec_mode != ExecMode.MASTER_NOTEBOOK, 'With MASTER_NOTEBOOK exec_mode one should not overwrite any of the launches (experiments)'
    LS.artifact_registry.attach_asset(lc.name, lc.version, lc.main_asset_fname, replace=True)
    meta = dict(
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='meta', replace=True)
else:
    assert CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK, 'MASTER_NOTEBOOK exec_mode is for working on 0 (dummy) version only'

LS.optuna_trial = None
optuna_trial_fname = os.path.join(CONFIG.initrd_path, 'optuna_trial.json')

if os.path.exists(optuna_trial_fname):
    with open(os.path.join(optuna_trial_fname), 'rt') as f:
        LS.optuna_trial = json.load(f)
        assert 'trial_number' in LS.optuna_trial, LS.optuna_trial
        assert 'study_serial' in LS.optuna_trial, LS.optuna_trial
        assert 'study_name' in LS.optuna_trial, LS.optuna_trial

    LOG(f'Optuna trial loaded from "{optuna_trial_fname}": {LS.optuna_trial}')

summary_log_dir = lc.name

if LS.optuna_trial is not None:
    summary_log_dir = os.path.join(summary_log_dir, f'opt_{LS.optuna_trial['study_serial']}')
    
summary_log_dir = os.path.join(summary_log_dir, str(lc.version))
LOG(f'Tensorboard run={summary_log_dir}')

summary_writer_type = os.environ.get('SUMMARY_WRITER', '').upper()

if not summary_writer_type:
    summary_writer_type = lu.when(CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK, 'S3', 'RMQ')

match summary_writer_type:
    case 'RMQ':
        kwargs = {}

        if os.environ.get('RMQ_CONNECTION_URL', None) is not None:
            kwargs['rmq_connection_url'] = os.environ['RMQ_CONNECTION_URL']

        LS.summary_writer = RmqSummaryWriter(log_dir=summary_log_dir, **kwargs)
        LOG(f'Created RmqSummaryWriter')
    case 'S3':
        LS.summary_writer = S3SummaryWriter(log_dir=summary_log_dir)
        LOG(f'Created S3SummaryWriter')
    case _:
        assert False, f'Unsupported {summary_writer_type=}'

LS.summary_writer.add_text('hyperparameters', pprint.pformat(HP._asdict(), sort_dicts=False), 0)
LS.summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 0)
LS.summary_writer.flush()

Random seed=42
torch.backends.cudnn.deterministic=True
Created ArtifactRegistry
Tensorboard run=18a_world_model_01/0
Created RmqSummaryWriter


# Agents

## load_agent

In [12]:
# @launchit.collect_explore
import launchit

In [13]:
# @launchit.collect_explore
# @launchit.disable_worker
def load_agent(agent_id, artifact_registry, create_instance=False):
    coords = hp_parse_artifact_source(agent_id)
    
    meta = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='meta', asset_ext='json', maven_group_id=coords.group_id)
    meta = json.loads(meta.decode('utf-8'))
    hp = meta['hypers']
    
    notebook_data = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='ipynb', maven_group_id=coords.group_id)
    source_code = launchit.extract_source_code(io.BytesIO(notebook_data), collect_inds=[None, 'explore'])
    module = lu.make_module(f'{agent_id}_notebook', source_code)
    module.CONFIG = CONFIG
    module.RNG = RNG
    module.LOG = LOG

    pt_data = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='agent', asset_ext='pt', maven_group_id=coords.group_id)
    agent_params = artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_classifier='agent_params', asset_ext='json', maven_group_id=coords.group_id)
    agent_params = module.Agent.Params(**json.loads(agent_params.decode('utf-8')))
    agent_params.observation_space_shape = tuple(agent_params.observation_space_shape)

    if not create_instance:
        return None, hp
        
    agent = module.Agent(agent_params)

    if hp['system']['is_torch_compile']:
        agent = torch.compile(agent, fullgraph=True)
        
    agent = agent.to(CONFIG.cuda_device)
    
    with io.BytesIO(pt_data) as b:
        kwargs = {}
    
        if not CONFIG.is_cuda:
            kwargs['map_location'] = torch.device('cpu')
        
        agent.load_state_dict(torch.load(b, **kwargs))

    return agent, hp

## Configure

In [14]:
# @launchit.disable
# @launchit.collect
HP.agents = [
    'com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162',
]
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True,
            'use_amp': True},
 'agents': ['com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162'],
 'model': {'d_model': 256,
           'layers_count': 3,
           'heads_count': 4,
           'ob_shape': (1, 84, 84),
           'ob_grid_shape': (16, 16),
           'sequence_length': 4,
           'attention_backend': 'MATH'},
 'rollout': {'steps_count': 512,
             'envs_count': 32,
             'env_rams': None,
             'env_ram_patches': None,
             'env_stories': ['*;*']},
 'train': {'test_pack': None,
           'epochs_count': 1,
           'batch_size': 32,
           'optimizer': 'AdamW',
           'max_grad_norm': 1.0,
           'learn_rate': 'const(0.00025)',
           'bce_loss_coef': 'const(1.0)',
           'edge_loss_coef': 'const(

## Initrd

In [15]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig' and HP.agents:
    for agent_id in HP.agents:
        load_agent(agent_id, CONFIG.artifact_registry, create_instance=False)
        LOG(f'Bootstrap data for agent "{agent_id}" loaded')

# @launchit.stop

## Create

In [16]:
# @launchit.disable_worker
LS.agents = {}
LS.agent_params = None

for agent_id in HP.agents:
    agent, agent_hp = load_agent(agent_id, LS.artifact_registry, create_instance=True)
    
    if not LS.agents:
        LS.env_id = agent_hp['env']['ident']
        LS.env_is_episodic_life = agent_hp['env']['is_episodic_life']
        LS.agent_params = agent.params
    else:
        assert LS.env_id == agent_hp['env']['ident'], (LS.env_id, agent_hp['env']['ident'])
        assert LS.env_is_episodic_life == agent_hp['env']['is_episodic_life'], (LS.env_is_episodic_life, agent_hp['env']['is_episodic_life'])
        assert LS.agent_params == agent.params, (LS.agent_params, agent.params)
    
    LS.agents[agent_id] = agent
    LOG(f'Agent {agent_id} loaded')

LOG(f'{LS.env_id=}')
LOG(f'{LS.env_is_episodic_life=}')
LOG(f'{LS.agent_params=}')
LOG(f'{len(LS.agents)=}')

Agent com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162 loaded
LS.env_id='FrostbiteNoFrameskip-v4'
LS.env_is_episodic_life=True
LS.agent_params=Agent.Params(d_model=256, cnn_expand_dim=1024, layers_count=3, heads_count=4, positional_encoding='learned', observation_space_shape=(3, 84, 84), actions_count=6, obs_sequence_length=4, action_plan_length=10)
len(LS.agents)=1


# Environment

## MyUberWrapper

In [17]:
# @launchit.collect_explore
import gymnasium as gym 
import ale_py

In [18]:
# @launchit.collect_explore
class MyUberWrapper(gym.vector.VectorWrapper):
    def __init__(self, env, idle_penalty, life_lost_penalty, frame_skip, track_levels):
        super().__init__(env)
        assert isinstance(env, ale_py.AtariVectorEnv)
        self.frame_numbers = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.episode_returns = np.zeros(env.unwrapped.num_envs)
        self.lives = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.was_life_lost = np.zeros(env.unwrapped.num_envs, dtype=np.bool)
        self.life_lengths = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
        self.life_returns = np.zeros(env.unwrapped.num_envs)
        assert idle_penalty <= 0
        assert life_lost_penalty <= 0
        self.idle_penalty = idle_penalty
        self.life_lost_penalty = life_lost_penalty
        self.frame_skip = frame_skip # forward data, used within video capture to compute fps
        self.track_levels = track_levels

        if self.track_levels:
            self.levels = np.zeros(env.unwrapped.num_envs, dtype=np.int32)
            self.temperatures = np.zeros(env.unwrapped.num_envs, dtype=np.uint8) 

    def step(self, actions):
        self.life_lengths[self.was_life_lost] = 0
        self.life_returns[self.was_life_lost] = 0
        
        obs, rewards, terminations, truncations, infos = self.env.step(actions)

        frame_numbers = infos['frame_number'] # grows indefinitely starting from ROM load or from state load
        episode_lengths = infos['episode_frame_number'] # resets on every new episode (game)
        lives = infos['lives']

        addon_lenghts = frame_numbers - self.frame_numbers
        self.frame_numbers[:] = frame_numbers
        assert np.all(addon_lenghts >= 0), addon_lenghts
        
        infos['is_episode_over'] = np.logical_or(terminations, truncations)
        self.episode_lengths[:] = episode_lengths
        self.episode_returns += rewards

        infos['is_life_lost'] = np.logical_or(lives < self.lives, infos['is_episode_over'])
        self.was_life_lost[:] = infos['is_life_lost']
        self.lives[:] = lives
        
        self.life_lengths += addon_lenghts
        self.life_returns += rewards

        infos['episode_lengths'] = self.episode_lengths.copy()
        infos['episode_returns'] = self.episode_returns.copy()
        infos['life_lengths'] = self.life_lengths.copy()
        infos['life_returns'] = self.life_returns.copy()

        clipped_rewards = np.sign(rewards)

        if self.idle_penalty < 0:
            clipped_rewards = np.where(clipped_rewards == 0, self.idle_penalty, clipped_rewards)

        if self.life_lost_penalty < 0:
            clipped_rewards[infos['is_life_lost']] = self.life_lost_penalty

        if self.track_levels:
            new_temperatures = infos['rams'][:,101].ravel()
            where_raised = new_temperatures > self.temperatures
            where_not_life_lost = ~infos['is_life_lost']
            where_level_passed = where_raised & where_not_life_lost
            self.levels[where_level_passed] += 1
            self.temperatures = new_temperatures
            infos['level_passed'] = where_level_passed

        return self._fix_obs(obs), clipped_rewards, terminations, truncations, infos

    def reset(self, seed=None, options=None):
        obs, infos = self.env.reset(seed=seed, options=options)
        reset_mask = lu.coalesce(options, {}).get('reset_mask')

        if reset_mask is None:
            reset_mask = np.full(len(self.episode_lengths), True, dtype=np.bool)

        self.frame_numbers[reset_mask] = infos['frame_number'][reset_mask]
        self.episode_lengths[reset_mask] = 0
        self.episode_returns[reset_mask] = 0
        self.life_lengths[reset_mask] = 0
        self.life_returns[reset_mask] = 0
        self.was_life_lost[reset_mask] = False

        if self.track_levels:
            self.levels[reset_mask] = 0
            self.temperatures[reset_mask] = infos['rams'][reset_mask,101].ravel()
            
        return self._fix_obs(obs), infos

    def _fix_obs(self, obs):
        # Get rid of dummy dimension cast by degenerate stack frames=1
        assert obs.ndim == 5, obs.shape # [num_envs, stack_size, height, width, 3]
        return obs.reshape(obs.shape[0], *obs.shape[2:])        

    def __repr__(self):
        return f'<{self.__class__.__name__}(idle_penalty={self.idle_penalty}, life_lost_penalty={self.life_lost_penalty}), {self.env}>'

## EnvRamPatcher

In [19]:
# @launchit.collect_explore
class EnvRamPatcher:
    def __call__(self, patch):
        assert patch is None or isinstance(patch, list)
        result = {}

        if patch is None:
            return result

        for p in patch:
            func = getattr(self, p)
            func(result)

        return result
    
    def no_score(self, ram):
        ram[73] = 0 
        ram[74] = 0
    
    def last_life(self, ram):
        ram[76] = 0

    def three_lives(self, ram):
        ram[76] = 3

    def eight_lives(self, ram):
        ram[76] = 8

    def nine_lives(self, ram):
        ram[76] = 9

    def full_igloo(self, ram):
        ram[77] = 15

    def half_igloo(self, ram):
        ram[77] = 7

    def one_remaining_igloo(self, ram):
        ram[77] = 14

    def three_remaining_igloo(self, ram):
        ram[77] = 12

    def no_igloo(self, ram):
        ram[77] = 255

    def temperature_10(self, ram):
        ram[101] = 10

    def temperature_20(self, ram):
        ram[101] = 32

    def bailey_right_at_the_igloo_door(self, ram):
        ram[102] = 124

    def bailey_very_near_igloo_door(self, ram):
        ram[102] = RNG.integers(108, 137, endpoint=True).item()

    def bailey_to_the_left_of_igloo(self, ram):
        ram[102] = RNG.integers(115, 125, endpoint=True).item()
    
    def bailey_to_the_right_of_igloo(self, ram):
        ram[102] = RNG.integers(130, 150, endpoint=True).item()

    def bailey_near_center(self, ram):
        ram[102] = RNG.integers(45, 95, endpoint=True).item()

    def bailey_random_spawn(self, ram):
        ram[102] = RNG.integers(16, 150, endpoint=True).item()

    def bailey_left_spawn(self, ram):
        ram[102] = 16

    def bailey_right_spawn(self, ram):
        ram[102] = 150

    def bear_to_the_left_of_igloo(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()

    def bear_chases_bailey_to_the_left_of_igloo(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()
        ram[102] = ram[104] + 25

    def bear_chases_bailey_to_the_left_of_igloo_2(self, ram):
        ram[104] = RNG.integers(25, 75, endpoint=True).item()
        ram[102] = 110

    def bear_chases_bailey_to_the_right_of_igloo(self, ram):
        ram[104] = RNG.integers(145, 150, endpoint=True).item()
        ram[102] = ram[104] - 15

## create_envs

In [20]:
# @launchit.collect_explore
def create_envs(env_id, envs_count, idle_penalty=None, life_lost_penalty=None, thread_pool_size=None, thread_affinity_offset=None, track_levels=False):
    assert 'NoFrameskip-v4' in env_id
    env_id = env_id[:env_id.index('NoFrameskip')].lower()
    frame_skip = 4
    
    envs = ale_py.AtariVectorEnv(
        game=env_id, 
        stack_num=1, # frame stacking is not needed because we use Transformer
        maxpool=True, # Combination of maxpool=True and 
        frameskip=frame_skip, # frameskip=4 corresponds to MaxAndSkipEnv(env, skip=4)
        repeat_action_probability=0, # corresponds NoFrameskip-v4
        use_fire_reset=True, # FireResetEnv(env)
        noop_max=30, # NoopResetEnv(env, noop_max=30)
        
        episodic_life=False, # will manage episodic life our selves
        life_loss_info=False, # strange parameter. When set to True then program will segfault
        reward_clipping=False, # will clip reward in wrapper in order to keep original rewards for game stats accounting

        # Get raw observation since rescaling and grayscaling is done on GPU
        grayscale=False,
        img_height=210,
        img_width=160,
        
        num_envs=envs_count, 
        num_threads=lu.coalesce(thread_pool_size, 0), # 0 means num of threads = num of envs
        thread_affinity_offset=lu.coalesce(thread_affinity_offset, -1), # -1 means no affinity

        autoreset_mode=gym.vector.vector_env.AutoresetMode.DISABLED,

        return_ram=track_levels,
    )

    return MyUberWrapper(
        envs, 
        idle_penalty=0, 
        life_lost_penalty=0,
        frame_skip=frame_skip, # used in video capturing to compute fps
        track_levels=track_levels,
    )

# WorldModel

## WorldModel

In [21]:
# @launchit.collect_explore
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as VF
from torch.nn.attention import SDPBackend, sdpa_kernel

In [22]:
# @launchit.collect_explore
class WorldModel(nn.Module):
    @dataclass(slots=True)
    class Params:
        d_model: int = None
        layers_count: int = None
        heads_count: int = None
        actions_count: int = None
        ob_shape: tuple = (1, 178, 152)
        ob_grid_shape: tuple = (16, 16)
        sequence_length: int = None
        attention_backend: str = None
        
    def __init__(self, params):
        super().__init__()
        self.params = params
        assert isinstance(self.params.ob_shape, tuple), self.params.ob_shape
        assert isinstance(self.params.ob_grid_shape, tuple), self.params.ob_grid_shape
        assert self.params.sequence_length >= 1, self.params.sequence_length
        self.patches_per_ob = self.params.ob_grid_shape[0] * self.params.ob_grid_shape[1] # e.g., 16x16=256
        self.ob_patches_count = self.patches_per_ob * self.params.sequence_length # e.g., 16x16x4=1024
        self.int_sequence_length = self.ob_patches_count + self.params.sequence_length # internal sequence length; second addendum is for each g var

        # @kms normalizations in between?
        if self.params.ob_shape == (1, 178, 152):
            self.cnn = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=8, stride=4, padding=2),
                nn.ReLU(),
                nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
                nn.ReLU(),
                nn.Conv2d(64, self.params.d_model, kernel_size=3, stride=1, padding=1),
                nn.ReLU(),
                # Guarantees a 16x16 feature map from a 178x152 input
                nn.AdaptiveAvgPool2d(self.params.ob_grid_shape),
            )
        else:
            assert False, f'Unsupported {params.ob_shape=}'

        self.post_cnn_ln = nn.LayerNorm(self.params.d_model)

        # kms@ what is better combined or separated encodings?
        # self.temporal_encoding = nn.Parameter(torch.randn(1, self.ob_sequence_length, self.params.d_model)) # seq number (0,1,2,3...) -> embedding
        # self.spatial_encoding = nn.Parameter(torch.randn(1, self.patches_per_ob, self.params.d_model)) # patch location -> embedding
        self.spatio_temporal_encoding = nn.Parameter(torch.randn(1, self.ob_patches_count, self.params.d_model)) # combined learned embedding for spatial and temporal location

        # kms@ simple nn.Embedding vs nn.Embedding + nn.Linear
        assert self.params.actions_count > 0
        self.action_encoding = nn.Sequential(
            nn.Embedding(self.params.actions_count, self.params.d_model),
            nn.Linear(self.params.d_model, self.params.d_model),
            nn.ReLU(),
        )
        # kms@ post_action_encoding normalization?

        # # world state latent variables - the main ingredient for prediction of next obs
        self.g_count = self.params.sequence_length
        # # kms@ single g instead of several? It makes sense to think of initial g embedding as a starting point to derive world model.
        # # As such it looks like there is no diff if we start to derive world model for 1, 2, 3 or 4 observations. The baseline is the same
        # # but available input data differ and hence accuracy of g will differ. In other words accuracy must depend only on data available and not on baseline
        # self.g_embs = nn.Parameter(torch.randn(1, self.g_count, self.params.d_model)) 
        self.g_encoding = nn.Parameter(torch.randn(1, 1, self.params.d_model)) 

        # kms@ alternative idea where g could attend to prev g
        causal_mask = self.generate_block_shaped_causal_mask()
        self.register_buffer('causal_mask', causal_mask)
        
        transformer_layer = nn.TransformerEncoderLayer(
            d_model=self.params.d_model, 
            dim_feedforward=self.params.d_model * 4, # kms@ default is 2048, 384 * 4 = 1536
            nhead=self.params.heads_count, 
            batch_first=True,
            norm_first=True,  # preferred for RL tasks
            dropout=0.0,      # Dropout destroys RL performance; keep 0.0
        )

        # One could think about transformer as a RNN on steroids. Each layer is an unrolled RNN step with its own weights.
        # And also it obtains Hebbian-like memory with perfect storage without noise (input sequence)
        self.transformer = nn.TransformerEncoder(
            transformer_layer, 
            num_layers=self.params.layers_count,
            enable_nested_tensor=False, # True is incompatible with layers where norm_first=True
        )
        self._init_transformer_weights(self.transformer)
        
        self.post_transformer_ln = nn.LayerNorm(self.params.d_model) 

        if self.params.ob_shape == (1, 178, 152):
            # kms@ maybe normalization LayerNorm/GroupNorm?
            self.transp_cnn_input_shape = (64, 11, 9)
            self.pre_transp_cnn = nn.Linear(self.params.d_model, math.prod(self.transp_cnn_input_shape))
            self.transp_cnn = nn.Sequential(
                nn.ConvTranspose2d(64, 48, kernel_size=4, stride=2, padding=1, output_padding=(0, 1)), # -> (48, 22, 19)
                nn.ReLU(),
                nn.ConvTranspose2d(48, 32, kernel_size=4, stride=2, padding=1), # -> (32, 44, 38)
                nn.ReLU(),
                nn.ConvTranspose2d(32, 16, kernel_size=4, stride=2, padding=1, output_padding=(1, 0)), # -> (16, 89, 76)
                nn.ReLU(),
                nn.ConvTranspose2d(16, 1, kernel_size=4, stride=2, padding=1), # -> 1, 178, 152)
                nn.Sigmoid(),  # Scales pixel intensities between 0.0 and 1.0
            )
        else:
            assert False, f'Unsupported {params.ob_shape=}'

        self.attention_backend = getattr(SDPBackend, self.params.attention_backend)

    @property
    def grad_norm_groups(self):
        return dict(
            cnn=list(self.cnn.parameters()),
            spatio_temporal_encoding=[self.spatio_temporal_encoding],
            action_encoding=list(self.action_encoding.parameters()),
            g_encoding=[self.g_encoding],
            transformer=list(self.transformer.parameters()),
            pre_transp_cnn=list(self.pre_transp_cnn.parameters()),
            transp_cnn=list(self.transp_cnn.parameters()),
        )

    def forward(self, obs, actions, padding_masks):
        batch_size = len(obs)
        
        x = self.embed_input_data(obs, actions)
        
        # Attach g tokens
        # g_embs = self.g_embs.expand(batch_size, -1, -1)
        # x = torch.cat([x, g_embs], dim=1) # (batch, ob_seq*patches_per_ob, d_model)->(batch, seq=ob_seq*(patches_per_ob+1), d_model)
        # assert x.shape[1] == self.int_sequence_length
        g_embs = self.g_encoding.expand(batch_size, self.g_count, -1)
        x = torch.cat([x, g_embs], dim=1) # (batch, ob_seq*patches_per_ob, d_model)->(batch, seq=ob_seq*(patches_per_ob+1), d_model)
        assert x.shape[1] == self.int_sequence_length

        # Explicitly force a specific backend (e.g., FlashAttention)
        with sdpa_kernel(self.attention_backend):
            # obs are expected to be left-padded, i.e. the very fresh obs is [:,-1] within each batch (env) is the last one

            # since each ob is exploded into patches_per_ob tokens we need to adjust padding_masks, e.g. 
            # t = torch.tensor([[1, 0, 1, 0], [1, 0, 0, 1]])
            # torch.repeat_interleave(t, 3, dim=1)
            # tensor([[1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
            #         [1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1]])
            assert padding_masks.shape == (batch_size, self.params.sequence_length)
            padding_masks = torch.repeat_interleave(padding_masks, self.patches_per_ob, dim=1)

            # g_embs are not pad masked so add zeros
            padding_masks = torch.cat([padding_masks, torch.zeros(batch_size, self.g_count, dtype=padding_masks.dtype, device=padding_masks.device)], dim=1)
            assert padding_masks.shape == (batch_size, self.int_sequence_length), padding_masks.shape

            y = self.transformer(
                x, 
                src_key_padding_mask=padding_masks,
                mask=self.causal_mask, 
                is_causal=True)
            
        y = self.post_transformer_ln(y)
        g = y[:,-self.g_count:] # grab last embs which correspond to calculated g-s

        predicted_obs = self.predict_next_obs(g)
        
        return predicted_obs

    def generate_block_shaped_causal_mask(self):
        # https://gist.github.com/kocherovms/0038eac144ca468792454fb6f5e544e5
        # Main idea: 
        # 1) patches within single observation could attend to each other and to patches from prev obs but cannot attend to patches from future obs
        # 2) g_0 token could attend to patches from first ob only and itself
        # 3) g_1 token could attend to patches from first and second obs only and itself
        # 4) g_i could attend to patches from [0:i+1] obs only and itself
        # Alternative idea:
        # 1) patches within single ob could attend to each other and to patches from prev obs but cannot attend to patches from future obs
        # 2) g_0 token could attend to patches from first ob only and itself
        # 3) g_1 token could attend to patches from first and second obs only, itself and g_0
        # 4) g_i could attend to patches from [0:i+1] obs only, itself and all previous g_j
        cm = nn.Transformer.generate_square_subsequent_mask(self.params.sequence_length)
        cm = torch.where(torch.isinf(cm), 1, 0)
        cm1 = torch.repeat_interleave(cm, self.patches_per_ob, dim=1)
        cm2 = torch.repeat_interleave(cm1, self.patches_per_ob, dim=0)
        cm3 = torch.ones(self.int_sequence_length, self.int_sequence_length)
        cm3[0:len(cm2),0:len(cm2)] = cm2
        cm3[len(cm2):,len(cm2):] = torch.where(torch.eye(self.g_count, self.g_count) == 1, 0, 1)
        cm3[len(cm2):,:len(cm2)] = cm1
        return torch.where(cm3 == 1, -torch.inf, cm3)

    def embed_input_data(self, obs, actions):
        # expected obs.shape = [batch, seq, color, height, width], i.e. batch with sequence of images in planar color layout
        # expected actions.shape = [batch, seq]
        assert obs.dtype == torch.uint8
        obs = obs / 255.0
        
        obs_shape = obs.shape
        assert len(obs_shape) == 5, len(obs_shape)
        assert tuple(obs_shape[-3:]) in [(1, 178, 152)], obss_shape
        
        batch_size = len(obs)
        assert batch_size == len(actions), (batch_size, len(actions))
        seq_len = obs.shape[1]
        assert seq_len == actions.shape[1], (seq_len, actions.shape[1])
            
        obs = obs.view(-1, *obs_shape[-3:]) # (uberbatch, color, height, width)
        ob_patch_embs = self.cnn(obs) # (uberbatch, d_model, 16, 16)
        ob_patch_embs = ob_patch_embs.permute(0, 2, 3, 1) # (uberbatch, 16, 16, d_model)
        ob_patch_embs = ob_patch_embs.reshape(-1, seq_len * self.patches_per_ob, self.params.d_model) # (batch, 16x16x4, d_model)
        ob_patch_embs = self.post_cnn_ln(ob_patch_embs)
        
        action_embs = self.action_encoding(actions) # (batch, seq, d_model)
        action_embs = action_embs.unsqueeze(2).expand(-1, -1, self.patches_per_ob, -1) # (batch, seq, 16x16, d_model)
        action_embs = action_embs.reshape(batch_size, self.ob_patches_count, self.params.d_model) # (batch, 16x16x4, d_model)

        assert ob_patch_embs.shape == action_embs.shape
        return ob_patch_embs + action_embs + self.spatio_temporal_encoding

    def predict_next_obs(self, g):
        # g.shape = (batch, g_count, d_model)
        g_shape = g.shape
        y = self.pre_transp_cnn(g) # (batch, g_count, 64 * 11 * 9)
        y = y.view(-1, *self.transp_cnn_input_shape) # defactorize last dim (uberbatch, 64, 11, 9)
        y = self.transp_cnn(y) # (uberbatch, self.params.ob_shape)
        return y.view(*g_shape[:2], *self.params.ob_shape)

    def preprocess_obs(self, obs):
        assert isinstance(obs, torch.Tensor)
        assert obs.dtype == torch.uint8
        obs_ndim = obs.ndim
        assert obs_ndim >= 3 # height, width, color

        if obs_ndim == 3:
            obs = obs.unsqueeze(0) # ensure there is a batch dim
        
        obs_shape = obs.shape
        obs = obs.view(-1, *obs_shape[-3:])
        obs = obs.permute(0, 3, 1, 2) # move color channel to the front (switch interleaved->planar format)
        
        if self.params.ob_shape == (1, 178, 152):
            # kms@ actually can get rid of top and bottom lines (they are black) -> (176, 152)
            obs = VF.rgb_to_grayscale(obs, num_output_channels=1)
            obs = obs[:,:,8:186,8:160] # extract only meaningful data for Frostbite
            obs = obs.view(*obs_shape[:-3], 1, 178, 152)
        elif self.params.ob_shape == (3, 84, 84):
            obs = VF.resize(obs, [84, 84], interpolation=VF.InterpolationMode.BILINEAR, antialias=True)
            obs = obs.view(*obs_shape[:-3], 3, 84, 84)
        elif self.params.ob_shape == (1, 84, 84):
            obs = VF.rgb_to_grayscale(obs, num_output_channels=1)
            obs = VF.resize(obs, [84, 84], interpolation=VF.InterpolationMode.BILINEAR, antialias=True)
            obs = obs.view(*obs_shape[:-3], 1, 84, 84)
        else:
            assert False, f'Unsupported {self.params.ob_shape=}'
        
        if obs_ndim == 3:
            obs = obs.squeeze(0)
        
        return obs

    @staticmethod
    def _init_weights(l, gain=np.sqrt(2), bias_const=0.0):
        '''
        CleanRL style orthogonal initialization helper
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        nn.init.orthogonal_(l.weight, gain=gain)
        
        if l.bias is not None:
            nn.init.constant_(l.bias, bias_const)
            
        return l

    @staticmethod
    def _init_transformer_weights(t):
        '''
        Applies orthogonal initialization to the inner transformer blocks.
        https://share.google/aimode/BDty0f9ZJZ6OXBFFQ
        '''
        
        for name, param in t.named_parameters():
            if 'weight' in name and param.dim() >= 2:
                # Appling gain=1.0 keeps variance stable across depth
                nn.init.orthogonal_(param, gain=1.0)
            elif 'bias' in name:
                nn.init.constant_(param, 0.0)

## Test

In [23]:
# @launchit.disable
t = lu.ScopedVars()
t.device = CONFIG.cuda_device
# t.device = 'cpu'

t.wmp = WorldModel.Params(
    d_model=256,
    layers_count=3,
    heads_count=4,
    actions_count=6,
    ob_shape=(1, 178, 152),
    ob_grid_shape=(16, 16),
    sequence_length=4,
    attention_backend='MATH',
)
t.model = WorldModel(t.wmp).to(t.device)
print(t.model)
t.params_count = sum(p.numel() for p in t.model.parameters())
print(f'{t.params_count=:_}')

t.envs_count = 2
t.steps_count = 10

t.obs = torch.zeros(t.envs_count, t.wmp.sequence_length, *t.wmp.ob_shape, dtype=torch.uint8).to(t.device)
t.pmasks = torch.zeros(t.envs_count, t.wmp.sequence_length).to(t.device)
t.actions = torch.zeros(t.envs_count, t.wmp.sequence_length, dtype=torch.int).to(t.device)

print(f'{t.obs.shape=}')
print(f'{t.pmasks.shape=}')
print(f'{t.actions.shape=}')

t.r = t.model(t.obs, t.actions, t.pmasks)

print(f'{t.r.shape=}')

WorldModel(
  (cnn): Sequential(
    (0): Conv2d(1, 32, kernel_size=(8, 8), stride=(4, 4), padding=(2, 2))
    (1): ReLU()
    (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (3): ReLU()
    (4): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): ReLU()
    (6): AdaptiveAvgPool2d(output_size=(16, 16))
  )
  (post_cnn_ln): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (action_encoding): Sequential(
    (0): Embedding(6, 256)
    (1): Linear(in_features=256, out_features=256, bias=True)
    (2): ReLU()
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-2): 3 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=1

## Configure

In [24]:
# @launchit.disable
# @launchit.collect
HP.model.d_model = 256
HP.model.layers_count = 3
HP.model.heads_count = 4
HP.model.ob_shape = (1, 178, 152)
HP.model.ob_grid_shape = (16, 16)
HP.model.sequence_length = 4
HP.model.attention_backend = 'FLASH_ATTENTION'
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True,
            'use_amp': True},
 'agents': ['com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162'],
 'model': {'d_model': 256,
           'layers_count': 3,
           'heads_count': 4,
           'ob_shape': (1, 178, 152),
           'ob_grid_shape': (16, 16),
           'sequence_length': 4,
           'attention_backend': 'FLASH_ATTENTION'},
 'rollout': {'steps_count': 512,
             'envs_count': 32,
             'env_rams': None,
             'env_ram_patches': None,
             'env_stories': ['*;*']},
 'train': {'test_pack': None,
           'epochs_count': 1,
           'batch_size': 32,
           'optimizer': 'AdamW',
           'max_grad_norm': 1.0,
           'learn_rate': 'const(0.00025)',
           'bce_loss_coef': 'const(1.0)',
           'edge_loss_c

## Create

In [25]:
# @launchit.disable_worker
wmp = WorldModel.Params(
    d_model=HP.model.d_model,
    layers_count=HP.model.layers_count,
    heads_count=HP.model.heads_count,
    actions_count=LS.agent_params.actions_count,
    ob_shape=HP.model.ob_shape,
    ob_grid_shape=HP.model.ob_grid_shape,
    sequence_length=HP.model.sequence_length,
    attention_backend=HP.model.attention_backend,
)

LS.world_model = WorldModel(wmp).to(CONFIG.cuda_device)
LOG(f'WorldModel created')

if HP.system.is_torch_compile:
    LS.world_model = torch.compile(LS.world_model, fullgraph=True)
    LOG(f'WorldModel compiled')

WorldModel created
WorldModel compiled


# Dataset

## Dataset

In [26]:
# @launchit.collect_explore
@dataclass(slots=True)
class Dataset:
    # Metadata
    sequence_length: int = None
    prologue_size: int = None

    # Main data
    pmasks: object = None # padding masks
    obs: object = None # WorldModel format
    actions: object = None
    dones: object = None
    env_inds: object = None
    step_inds: object = None
    
    # IPC tribute
    shared_tensors: dict = None
    
    # Iteration support
    valid_inds: object = None
    batch_size: int = None
    is_shuffled: bool = None
    iter_inds: object = None
    iter_pos: int = None
    flattened: tuple = None
    selected: tuple = None
    bw_inds_stem: object = None

    Batch = namedtuple('Batch', 'pmasks, obs, actions, dones, env_inds, step_inds, b_inds')
    
    def __init__(
        self,
        envs_count, 
        ob_shape, 
        rollout_steps_count, 
        sequence_length,
        actions_count,
        batches_count,
        is_shuffled,
    ):
        self.sequence_length = sequence_length
        self.prologue_size = (sequence_length - 1)
        rollout_buf_size = self.prologue_size + rollout_steps_count # prologue (mem window from prev rollout) + actual rollout data
        assert rollout_buf_size > 0
        
        self.pmasks = torch.zeros((envs_count, rollout_buf_size, sequence_length)).to(CONFIG.cuda_device) 
        self.obs = torch.zeros((envs_count, rollout_buf_size, *ob_shape), dtype=torch.uint8).to(CONFIG.cuda_device)
        self.actions = torch.zeros((envs_count, rollout_buf_size, sequence_length), dtype=torch.long).to(CONFIG.cuda_device)
        self.dones = torch.zeros((envs_count, rollout_buf_size, sequence_length)).to(CONFIG.cuda_device)

        # a-la meshgrid to easily get pairs (env_ind, step_ind) by b_inds
        self.env_inds = torch.arange(envs_count).unsqueeze(1).expand(-1, rollout_buf_size).to(CONFIG.cuda_device)
        self.step_inds = torch.arange(-self.prologue_size, rollout_steps_count).unsqueeze(0).expand(envs_count, -1).to(CONFIG.cuda_device)
        assert self.env_inds.shape == self.step_inds.shape, (self.env_inds.shape, self.step_inds.shape)

        self.shared_tensors = dict(
            pmasks=self.pmasks,
            obs=self.obs,
            actions=self.actions,
            dones=self.dones,
            env_inds=self.env_inds,
            step_inds=self.step_inds,
        )
        
        for t in self.shared_tensors.values():
            if t is not None:
                t.share_memory_()

        # Iteration support
        valid_inds = []
        
        for env_ind in range(envs_count):
            offset = env_ind * rollout_buf_size + self.prologue_size
            indices_for_env = offset + torch.arange(rollout_steps_count)
            valid_inds.append(indices_for_env)
            
        self.valid_inds = torch.cat(valid_inds).to(CONFIG.cuda_device)
        assert envs_count * rollout_steps_count % batches_count == 0, f'Not whole number of batches for given rollout data size: {((envs_count * rollout_steps_count) / batches_count)=}'
        self.batch_size = envs_count * rollout_steps_count // batches_count
        self.is_shuffled = is_shuffled

        self.flattened = Dataset.Batch(
            pmasks=self.pmasks.view(-1, *self.pmasks.shape[2:]),
            obs=self.obs.view(-1, *self.obs.shape[2:]),
            actions=self.actions.view(-1, *self.actions.shape[2:]),
            dones=self.dones.view(-1, *self.dones.shape[2:]),
            env_inds=self.env_inds.reshape(-1, *self.env_inds.shape[2:]),
            step_inds=self.step_inds.reshape(-1, *self.step_inds.shape[2:]),
            b_inds=None,
        )
        self.selected = Dataset.Batch(
            pmasks=torch.ones(self.batch_size, sequence_length).to(CONFIG.cuda_device), 
            obs=torch.zeros(self.batch_size * sequence_length, *ob_shape, dtype=torch.uint8).to(CONFIG.cuda_device),
            actions=torch.zeros(self.batch_size, sequence_length, dtype=torch.long).to(CONFIG.cuda_device),
            dones=torch.zeros(self.batch_size, sequence_length).to(CONFIG.cuda_device),
            env_inds=torch.zeros(self.batch_size, dtype=torch.long).to(CONFIG.cuda_device),
            step_inds=torch.zeros(self.batch_size, dtype=torch.long).to(CONFIG.cuda_device),
            b_inds=None,
        )
        self.bw_inds_stem = torch.arange(-sequence_length + 1, 1).to(CONFIG.cuda_device)
        self.bw_inds_stem = self.bw_inds_stem.unsqueeze(0).expand(self.batch_size, self.bw_inds_stem.shape[0])

    def __iter__(self):
        if self.is_shuffled:
            valids_inds_order = torch.randperm(len(self.valid_inds), device=CONFIG.cuda_device)
            self.iter_inds = self.valid_inds[valids_inds_order]
        else:
            self.iter_inds = self.valid_inds

        self.iter_pos = 0
        return self
    
    def __next__(self):
        if self.iter_pos >= len(self.iter_inds):
            raise StopIteration

        b_inds = self.iter_inds[self.iter_pos:self.iter_pos+self.batch_size]
        self.iter_pos += self.batch_size

        # windows stored as is
        index = b_inds.unsqueeze(1).expand((-1, self.sequence_length))
        torch.gather(input=self.flattened.pmasks, index=index, dim=0, out=self.selected.pmasks)
        torch.gather(input=self.flattened.actions, index=index, dim=0, out=self.selected.actions)
        torch.gather(input=self.flattened.dones, index=index, dim=0, out=self.selected.dones)
        # non-windows
        torch.index_select(input=self.flattened.env_inds, index=b_inds, dim=0, out=self.selected.env_inds)
        torch.index_select(input=self.flattened.step_inds, index=b_inds, dim=0, out=self.selected.step_inds)
        # reconstructed windows (may not be stored as is due to memory limits)
        bw_inds = self.bw_inds_stem + b_inds.unsqueeze(1) # batch-window indices, shape: [batch, seq_len]
        bw_inds = bw_inds.view(-1)
        torch.index_select(input=self.flattened.obs, index=bw_inds, dim=0, out=self.selected.obs)
        
        return Dataset.Batch(
            pmasks=self.selected.pmasks,
            obs=self.selected.obs.view((self.batch_size, self.sequence_length, *self.selected.obs.shape[1:])),
            actions=self.selected.actions,
            dones=self.selected.dones,
            env_inds=self.selected.env_inds,
            step_inds=self.selected.step_inds,
            b_inds=b_inds,
        )

## Test

### Smoke test

In [27]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 8
t.rollout_steps_count = 64
t.sequence_length = 10
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=(3, 84, 84), 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    actions_count=t.actions_count,
    batches_count=2,
    is_shuffled=False,
)

for t.name, t.v in t.dataset.shared_tensors.items():
    if t.v is not None:
        assert t.v.is_shared(), f'{t.name} is not shared'
        LOG(f'{t.name:>38}: {str(t.v.device):>6}, {str(t.v.dtype):>15}, {t.v.shape}')

assert len(t.dataset.valid_inds) == (t.envs_count * t.rollout_steps_count)
assert len(t.dataset.valid_inds.unique()) == (t.envs_count * t.rollout_steps_count)

t.prev_env_valid_inds = None

for t.env_ind in range(t.envs_count):
    t.prev_env_training_ind = lu.when(t.prev_env_valid_inds is not None, lambda: t.prev_env_valid_inds[-1] + 1, 0)
    t.env_valid_inds = t.prev_env_training_ind + (t.sequence_length - 1) + torch.arange(t.rollout_steps_count).to(CONFIG.cuda_device)
    assert torch.all(t.env_valid_inds == t.dataset.valid_inds[t.env_ind*t.rollout_steps_count:(t.env_ind+1)*t.rollout_steps_count])
    t.prev_env_valid_inds = t.env_valid_inds

                                pmasks:    cpu,   torch.float32, torch.Size([8, 73, 10])
                                   obs:    cpu,     torch.uint8, torch.Size([8, 73, 3, 84, 84])
                               actions:    cpu,     torch.int64, torch.Size([8, 73, 10])
                                 dones:    cpu,   torch.float32, torch.Size([8, 73, 10])
                              env_inds:    cpu,     torch.int64, torch.Size([8, 73])
                             step_inds:    cpu,     torch.int64, torch.Size([8, 73])


### Iteration

In [28]:
# @launchit.disable
t = lu.ScopedVars()
t.envs_count = 32
t.rollout_steps_count = 128
t.sequence_length = 10
t.actions_count = 6
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=(3, 84, 84), 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    actions_count=t.actions_count,
    batches_count=4,
    is_shuffled=True,
)

for t.batch in t.dataset:
    pass

for t.fn in t.batch._fields:
    if getattr(t.batch, t.fn) is not None:
        LOG(f'{t.fn:>20}.shape={getattr(t.batch, t.fn).shape}')
    else:
        LOG(f'{t.fn:>20}=None')

              pmasks.shape=torch.Size([1024, 10])
                 obs.shape=torch.Size([1024, 10, 3, 84, 84])
             actions.shape=torch.Size([1024, 10])
               dones.shape=torch.Size([1024, 10])
            env_inds.shape=torch.Size([1024])
           step_inds.shape=torch.Size([1024])
              b_inds.shape=torch.Size([1024])


# Rollout

## RolloutEnvStorySampler

In [37]:
class RolloutEnvStorySampler:
    def __init__(self, rams, ram_patches, stories):
        assert rams is None or isinstance(rams, dict)
        assert ram_patches is None or isinstance(ram_patches, list)
        assert isinstance(stories, list)
        assert stories
        self.rams = rams
        self.ram_patches = ram_patches
        self.ram_patcher = EnvRamPatcher()
        self.stories = stories

        if self.ram_patches is not None:
            # Verify for all atomic ram_patch there is a ram_patcher func
            for atomic_ram_patch in set(itertools.chain.from_iterable(self.ram_patches)):
                getattr(self.ram_patcher, atomic_ram_patch)
 
    EnvStory = namedtuple('EnvStory', 'ram, ram_patch, desc')

    def sample(self):
        desc = []
        
        story = RNG.choice(self.stories).item()
        avl_rams, avl_ram_patches = story.split(';')
        assert avl_rams, story
        assert avl_ram_patches, story

        if self.rams is None or not self.rams:
            ram = None
            desc.append('None')
        else:
            ram = self.pick(list(self.rams.keys()), avl_rams)
            desc.append(str(ram))
            ram = self.rams[ram]

        ram_patch = self.pick(self.ram_patches, avl_ram_patches)

        if ram_patch is None or not ram_patch:
            ram_patch = None
            desc.append('None')
        else:
            desc.append('+'.join(ram_patch))

        return RolloutEnvStorySampler.EnvStory(ram=ram, ram_patch=ram_patch, desc='/'.join(desc))

    @staticmethod
    def pick(l, inds):
        if l is None or not l:
            return None
            
        subl = None
        
        if inds == '*':
            subl = l
        elif ':' in inds:
            slice_inds = inds.split(':')
            assert len(slice_inds) == 2, inds
            subl = l[slice(int(slice_inds[0]), int(slice_inds[1]))]
        else:
            enum_inds = inds.split(',')
            subl = []
            
            for ind in enum_inds:
                subl.append(l[int(ind)])

        i = RNG.choice(len(subl))
        return subl[i]

## RolloutManager

In [87]:
# @launchit.collect_explore
class RolloutManager:
    def __init__(self, agent_sequence_length, agent_ob_shape, sequence_length, ob_shape, env_id, envs_count, random_seed, is_episodic_life=True, device=None):
        self.envs_count = envs_count
        self.random_seed = random_seed
        self.is_episodic_life = is_episodic_life
        self.device = lu.coalesce(device, CONFIG.cuda_device)

        # Sliding windows of an agent
        self.aw_pmasks = torch.ones((envs_count, agent_sequence_length)).to(self.device)
        self.aw_obs = torch.zeros((envs_count, agent_sequence_length, *agent_ob_shape), dtype=torch.uint8).to(self.device)
        
        # Sliding windows of a world model
        self.mw_pmasks = torch.ones((envs_count, sequence_length)).to(self.device)
        self.mw_obs = torch.zeros((envs_count, sequence_length, *ob_shape), dtype=torch.uint8).to(self.device)
        self.mw_actions = torch.zeros(envs_count, sequence_length, dtype=torch.long).to(self.device)
        self.mw_dones = torch.zeros(envs_count, sequence_length).to(self.device)
        
        thread_pool_size = lu.coalesce_fn(os.environ.get('ALE_THREAD_POOL_SIZE'), int, None)
        thread_affinity_offset = lu.coalesce_fn(os.environ.get('ALE_THREAD_AFFINITY_OFFSET'), int, None)
        self.envs = create_envs(env_id, envs_count, thread_pool_size=thread_pool_size, thread_affinity_offset=thread_affinity_offset)
        LOG(f'Envs created: {envs_count} envs, {thread_pool_size=}, {thread_affinity_offset=}')

        self.env_ram_patcher = EnvRamPatcher()
        self.is_reset = False

    def reset(self, agent, world_model, ram=None, ram_patch=None):
        reset_options = {}
        reset_options.update(lu.when(ram is not None, lambda: dict(rams={-1: ram}), {}))
        reset_options.update(lu.when(ram_patch is not None, lambda: dict(ram_patches=self.get_ram_patches(ram_patch, torch.arange(self.envs_count))), {}))
        raw_obs, _ = self.envs.reset(seed=self.random_seed, options=reset_options)
        raw_obs = torch.tensor(raw_obs, device=self.device)
        
        self.a_obs = agent.preprocess_obs(raw_obs)
        self.aw_obs[:,-1] = self.a_obs
        self.aw_pmasks[:,-1] = 0
        
        self.m_obs = world_model.preprocess_obs(raw_obs)
        self.mw_obs[:,-1] = self.m_obs
        self.mw_pmasks[:,-1] = 0

        self.is_reset = True
        LOG('Envs reset')
        
    def rollout(self, agent, world_model, dataset, steps_count, ram=None, ram_patch=None, with_timing_counters=False, with_control_data=False):
        if not self.is_reset:
            self.reset(agent, world_model, ram, ram_patch)
            assert self.is_reset
        
        out_pmasks, out_obs, out_actions, out_dones = (
            dataset.shared_tensors['pmasks'],
            dataset.shared_tensors['obs'],
            dataset.shared_tensors['actions'],
            dataset.shared_tensors['dones'],
        )
        assert steps_count + dataset.prologue_size == out_pmasks.shape[1]
        assert torch.all(self.mw_obs[:,-1] == self.m_obs)
        assert torch.all(self.aw_obs[:,-1] == self.a_obs)
        
        # fill prologues with data from the prev rollout
        # For obs prologue is mandatory since obs window is reconstructed. For others - only to keep shape the same
        out_obs[:,:dataset.prologue_size] = out_obs[:,-dataset.prologue_size:]

        control_data = defaultdict(list)
        rollout_counters = defaultdict(float)
        timing_counters = defaultdict(list)

        with torch.no_grad():
            for rollout_step in range(steps_count):
                t0 = time.time()
                
                out_index = dataset.prologue_size + rollout_step
                out_pmasks[:,out_index] = self.mw_pmasks
                out_obs[:,out_index] = self.m_obs
                
                if with_control_data:
                    control_data['pmasks'].append(self.mw_pmasks.cpu().numpy().copy())
                    control_data['obs'].append(self.mw_obs.cpu().numpy().copy())

                timing_counters['out_obs'].append(time.time() - t0)
                t0 = time.time()

                # Analyze observation windows
                agent_result = agent(
                    obs=self.aw_obs, 
                    padding_masks=self.aw_pmasks,
                )

                timing_counters['get_action_and_value'].append(time.time() - t0)
                t0 = time.time()
                
                # Interact with environments
                raw_obs, rewards, terminations, truncations, infos = self.envs.step(agent_result.actions.cpu().numpy().ravel())
                rollout_counters['reward'] += rewards.sum().item()
                
                raw_obs = torch.tensor(raw_obs, device=self.device)

                # Update sliding windows which are used to feed agent
                self.a_obs = agent.preprocess_obs(raw_obs)
                self.aw_obs = self.aw_obs.roll(shifts=-1, dims=1)
                self.aw_obs[:,-1] = self.a_obs
                self.aw_pmasks = self.aw_pmasks.roll(shifts=-1, dims=1)
                self.aw_pmasks[:,-1] = 0
                
                # Update sliding windows which are used to feed world model
                self.m_obs = world_model.preprocess_obs(raw_obs)
                self.mw_obs = self.mw_obs.roll(shifts=-1, dims=1)
                self.mw_obs[:,-1] = self.m_obs
                self.mw_pmasks = self.mw_pmasks.roll(shifts=-1, dims=1)
                self.mw_pmasks[:,-1] = 0
                self.mw_actions = self.mw_actions.roll(shifts=-1, dims=1)
                self.mw_actions[:,-1] = agent_result.actions

                out_actions[:,out_index] = self.mw_actions

                if with_control_data:
                    control_data['actions'].append(self.mw_actions.cpu().numpy().copy()) # copy is a must since on CPU torch could return shared data

                need_reset_envs = np.logical_or(terminations, truncations)
                dones = need_reset_envs.copy()

                if self.is_episodic_life and np.any(infos['is_life_lost']):
                    # Manually trigger done flag and truncate observations window (i.e. leave only current observation for new life)
                    # for envs which hit life loss
                    life_lost_envs = infos['is_life_lost']
                    # life_lost_envs must include need_reset_envs, i.e. it must be wider (superset)
                    assert np.all((need_reset_envs & life_lost_envs) == need_reset_envs), (need_reset_envs, life_lost_envs)
                    dones[life_lost_envs] = True
                    rollout_counters['fake_resets'] += life_lost_envs.sum().item()

                self.mw_dones = self.mw_dones.roll(shifts=-1, dims=1)
                self.mw_dones[:,-1] = torch.tensor(dones, dtype=torch.float).to(self.device, non_blocking=True)
                
                out_dones[:,out_index] = self.mw_dones

                if with_control_data:
                    control_data['dones'].append(self.mw_dones.cpu().numpy().copy()) # copy is a must since on CPU torch could return shared data
                
                timing_counters['env.step'].append(time.time() - t0)
        
                # Reset envs which hit a REAL episode end, for resetted envs get new observations
                if np.any(need_reset_envs):
                    t00 = time.time()
                    reset_options = dict(reset_mask=need_reset_envs)
                    reset_options.update(lu.when(ram is not None, lambda: dict(rams={-1: ram}), {}))
                    reset_options.update(lu.when(ram_patch is not None, lambda: dict(ram_patches=self.get_ram_patches(ram_patch, np.flatnonzero(need_reset_envs))), {}))
                    reset_obs, _ = self.envs.reset(options=reset_options)
                    reset_obs = torch.tensor(reset_obs[need_reset_envs]).to(self.device)
                    
                    self.a_obs[need_reset_envs] = agent.preprocess_obs(reset_obs)
                    self.aw_obs[:,-1] = self.a_obs
                    
                    self.m_obs[need_reset_envs] = world_model.preprocess_obs(reset_obs)
                    self.mw_obs[:,-1] = self.m_obs
                    
                    timing_counters['env.reset'].append(time.time() - t00)
                    rollout_counters['true_resets'] += need_reset_envs.sum().item()

                if np.any(dones):
                    self.aw_pmasks[dones] = 1
                    self.aw_pmasks[dones,-1] = 0
                    
                    self.mw_pmasks[dones] = 1
                    self.mw_pmasks[dones,-1] = 0
                    self.mw_actions[dones] = 0
                    self.mw_dones[dones] = 0

        return dict(
            rollout_counters=rollout_counters,
            timing_counters=lu.when(with_timing_counters, timing_counters, None),
            control_data=lu.when(with_control_data, control_data, None),
        )

    def get_ram_patches(self, ram_patch, env_inds):
        # create RAM patch for each env individually since RAM patch may include random effects 
        # otherwise ({-1: ram_patch}) we would get simlar environments and agent will learn slowly
        ram_patches = {}
        
        for env_ind in env_inds:
            ram_patches[int(env_ind)] = self.env_ram_patcher(ram_patch)

        return ram_patches

## Test

### rollout: pmasks, obs, actions, dones

In [39]:
# @launchit.disable
t = lu.ScopedVars()
t.agent = next(iter(LS.agents.values()))
t.sequence_length = LS.world_model.params.sequence_length + 1
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.df_columns = defaultdict(list)
t.all_life_stats_items = []
t.all_episode_stats_items = []
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=LS.world_model.params.ob_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    actions_count=t.agent.params.actions_count,
    batches_count=4,
    is_shuffled=False,
)
t.rm = RolloutManager(
    agent_sequence_length=t.agent.params.obs_sequence_length, 
    agent_ob_shape=t.agent.params.observation_space_shape,
    sequence_length=t.sequence_length,
    ob_shape=LS.world_model.params.ob_shape,
    env_id=LS.env_id,
    envs_count=t.envs_count, 
    random_seed=HP.system.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 5
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.agent, LS.world_model, t.dataset, steps_count=t.rollout_steps_count, with_control_data=True)

        t.control_pmasks = torch.tensor(np.array(t.rr['control_data']['pmasks']))
        t.control_obs = torch.tensor(np.array(t.rr['control_data']['obs']))
        t.control_actions = torch.tensor(np.array(t.rr['control_data']['actions']))
        t.control_dones = torch.tensor(np.array(t.rr['control_data']['dones']))
        assert len(t.control_pmasks) == t.rollout_steps_count
        assert len(t.control_obs) == t.rollout_steps_count
        assert len(t.control_actions) == t.rollout_steps_count
        assert len(t.control_dones) == t.rollout_steps_count
        
        # Verify that we get exactly the same windows (pmasks, obs, actions, dones) as during rollout
        # Also make sure important conditions are kept
        for t.batch in t.dataset:
            t.b_orig_w_pmasks = t.control_pmasks[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_pmasks == t.batch.pmasks)
            # current items are LAST items and are always unmasked, make sure pmasks reflects this fact
            assert torch.all(t.batch.pmasks[:,-1] == 0) 
            # make sure there are no pmasks with all ones
            t.pmasks_counts = t.batch.pmasks.sum(axis=1)
            assert torch.all(t.pmasks_counts < t.sequence_length)

            # make sure that zeros (unmasking) are pushed to pmasks right to left
            for t.l in range(1, t.sequence_length): # 1,2,..,sequence_length-1
                t.expected_pmask = torch.zeros(t.sequence_length, dtype=t.batch.pmasks.dtype).to(t.batch.pmasks.device)
                t.expected_pmask[:t.l] = 1
                t.seq_len_mask = (t.pmasks_counts == t.l)
                assert torch.all(t.batch.pmasks[t.seq_len_mask] == t.expected_pmask)
            
            t.b_orig_w_obs = t.control_obs[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_obs == t.batch.obs)

            t.b_orig_w_actions = t.control_actions[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_actions == t.batch.actions)

            t.b_orig_w_dones = t.control_dones[t.batch.step_inds, t.batch.env_inds]
            assert torch.all(t.b_orig_w_dones == t.batch.dones)

            # Since dones is a true windowed object (as opposed to obs which is a reconstructed one) 
            # then we demand that it doesn't contain leftovers from prev runs (episodes). 
            # As such if current item (last item in sequence) is done then all prev dones in this window must be 0
            t.dones_counts = t.batch.dones.sum(axis=1)
            assert torch.all(t.dones_counts <= 1)
            t.any_dones_mask = (t.dones_counts > 0)
            assert torch.all(t.batch.dones[t.any_dones_mask,-1] == 1)
            assert torch.all(t.batch.dones[t.any_dones_mask,:-1] == 0)
            # For any done env all items must be present (pmask)
            assert torch.all(t.batch.pmasks[t.any_dones_mask,:-1] == 0)

        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

        for t.x in ['reward', 'fake_resets', 'true_resets']:
            t.df_columns[t.x].append(t.rr['rollout_counters'][t.x])

pd.DataFrame(t.df_columns).style.format("{:.2f}")

Envs created: 8 envs, thread_pool_size=None, thread_affinity_offset=None


  0%|          | 0/5120 [00:00<?, ?it/s]

Envs reset


,reward,fake_resets,true_resets
0,101.00,0.00,0.00
1,47.00,8.00,0.00
2,364.00,0.00,0.00
3,148.00,0.00,0.00
4,202.00,4.00,0.00


### rollout: performance

In [40]:
# @launchit.disable
t = lu.ScopedVars()
t.agent = next(iter(LS.agents.values()))
t.sequence_length = LS.world_model.params.sequence_length + 1
t.envs_count = lu.when(CONFIG.is_cuda, 32, 8)
t.timing_counters = defaultdict(list)
t.rollout_steps_count = 128
t.dataset = Dataset(
    envs_count=t.envs_count,
    ob_shape=LS.world_model.params.ob_shape, 
    rollout_steps_count=t.rollout_steps_count, 
    sequence_length=t.sequence_length, 
    actions_count=t.agent.params.actions_count,
    batches_count=4,
    is_shuffled=True,
)
t.rm = RolloutManager(
    agent_sequence_length=t.agent.params.obs_sequence_length, 
    agent_ob_shape=t.agent.params.observation_space_shape,
    sequence_length=t.sequence_length,
    ob_shape=LS.world_model.params.ob_shape,
    env_id=LS.env_id,
    envs_count=t.envs_count, 
    random_seed=HP.system.random_seed,
)

t.global_steps_count = t.rollout_steps_count * t.envs_count * 10
t.step = 0

with tqdm(total=t.global_steps_count) as pbar:
    while t.step < t.global_steps_count:
        t.rr = t.rm.rollout(t.agent, LS.world_model, t.dataset, steps_count=t.rollout_steps_count, with_timing_counters=True)

        for t.timing_counter_key, t.timing_counter_times in t.rr['timing_counters'].items():
            t.timing_counters[t.timing_counter_key].extend(t.timing_counter_times)

        step_inc = t.rollout_steps_count * t.envs_count
        t.step += step_inc
        pbar.update(step_inc)

list(map(lambda kv: (kv[0], np.array(kv[1]).mean().item()), t.timing_counters.items()))

Envs created: 8 envs, thread_pool_size=None, thread_affinity_offset=None


  0%|          | 0/10240 [00:00<?, ?it/s]

Envs reset


[('out_obs', 5.357861518859863e-05),
 ('get_action_and_value', 0.01165652610361576),
 ('env.step', 0.006214245222508907),
 ('env.reset', 0.012921810150146484)]

## Configure

In [99]:
# @launchit.disable
# @launchit.collect
HP.rollout.steps_count = 128
HP.rollout.envs_count = 1
HP.rollout.env_rams = [
    'com.develorium.neurolab.frostbite_ram:level1_101:1',
    'com.develorium.neurolab.frostbite_ram:level4_101:1',
]
HP.rollout.env_ram_patches = None
HP.rollout.env_stories = ['*;*']
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True,
            'use_amp': True},
 'agents': ['com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162'],
 'model': {'d_model': 256,
           'layers_count': 3,
           'heads_count': 4,
           'ob_shape': (1, 178, 152),
           'ob_grid_shape': (16, 16),
           'sequence_length': 4,
           'attention_backend': 'FLASH_ATTENTION'},
 'rollout': {'steps_count': 128,
             'envs_count': 1,
             'env_rams': ['com.develorium.neurolab.frostbite_ram:level1_101:1',
                          'com.develorium.neurolab.frostbite_ram:level4_101:1'],
             'env_ram_patches': None,
             'env_stories': ['*;*']},
 'train': {'test_pack': 'test_pack:9',
           'epochs_count': 4,
           'batch_size': 32,
           'optimizer': 'AdamW',
   

## Initrd

In [100]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig' and HP.rollout.env_rams:
    for ram_id in HP.rollout.env_rams:
        coords = hp_parse_artifact_source(ram_id)
        CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', maven_group_id=coords.group_id)
# @launchit.stop

## Create

In [101]:
# @launchit.disable_worker

# @kms apply ram and ram_patch immediately
LS.rollout_manager = RolloutManager(
    agent_sequence_length=LS.agent_params.obs_sequence_length, 
    agent_ob_shape=LS.agent_params.observation_space_shape,
    sequence_length=LS.world_model.params.sequence_length + 1,
    ob_shape=LS.world_model.params.ob_shape,
    env_id=LS.env_id, 
    envs_count=HP.rollout.envs_count, 
    is_episodic_life=LS.env_is_episodic_life,
    random_seed=HP.system.random_seed,
)
LOG(f'RolloutManager created')

rollout_env_rams = None

if HP.rollout.env_rams:
    rollout_env_rams = {}
    
    for ram_id in HP.rollout.env_rams:
        coords = hp_parse_artifact_source(ram_id)
        ram = LS.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', maven_group_id=coords.group_id)

        with io.BytesIO(ram) as b:
            rollout_env_rams[ram_id] = pickle.load(b)

LS.rollout_env_story_sampler = RolloutEnvStorySampler(rollout_env_rams, HP.rollout.env_ram_patches, HP.rollout.env_stories)
LOG(f'RolloutEnvStorySampler created')

Envs created: 1 envs, thread_pool_size=None, thread_affinity_offset=None
RolloutManager created
RolloutEnvStorySampler created


# TRAIN

## EdgeDetector

In [102]:
# @launchit.disable_worker
# https://share.google/aimode/910bVkcGs9nvCh9Q8
class EdgeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        # Sobel filters to extract horizontal and vertical edges
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
        # Register as buffers so they move to the correct GPU automatically
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    ForwardResult = namedtuple('ForwardResult', 'grad_x, grad_y, magnitude')    
    
    def forward(self, x):
        # x shape: (Batch, 1, H, W)
        padded = F.pad(x, (1, 1, 1, 1), mode='replicate') # Pad edges to keep sizes matching
        grad_x = F.conv2d(padded, self.sobel_x)
        grad_y = F.conv2d(padded, self.sobel_y)
        magnitude = torch.sqrt(grad_x ** 2 + grad_y ** 2 + 1e-6)
        return EdgeDetector.ForwardResult(grad_x=grad_x, grad_y=grad_y, magnitude=magnitude)

## Configure

In [104]:
# @launchit.disable
# @launchit.collect
HP.train.test_pack = 'test_pack:9'
HP.train.epochs_count = 4
HP.train.batch_size = 32
HP.train.optimizer = 'AdamW'
HP.train.max_grad_norm = 1.0
HP.train.learn_rate = 'const(0.00025)'
HP.train.bce_loss_coef = 'const(1.0)'
HP.train.edge_loss_coef = 'const(1.0)'
# @launchit.stop

In [105]:
# @launchit.disable_worker
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'launch_goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'launch_id': 0,
 'system': {'comment': None,
            'random_seed': 42,
            'is_torch_deterministic': True,
            'is_torch_compile': True,
            'use_amp': True},
 'agents': ['com.develorium.neurolab.17_rl:17e_ppo_tr_atari_mp_16:162'],
 'model': {'d_model': 256,
           'layers_count': 3,
           'heads_count': 4,
           'ob_shape': (1, 178, 152),
           'ob_grid_shape': (16, 16),
           'sequence_length': 4,
           'attention_backend': 'FLASH_ATTENTION'},
 'rollout': {'steps_count': 128,
             'envs_count': 1,
             'env_rams': ['com.develorium.neurolab.frostbite_ram:level1_101:1',
                          'com.develorium.neurolab.frostbite_ram:level4_101:1'],
             'env_ram_patches': None,
             'env_stories': ['*;*']},
 'train': {'test_pack': 'test_pack:9',
           'epochs_count': 4,
           'batch_size': 32,
           'optimizer': 'AdamW',
   

## Initrd

In [106]:
# @launchit.collect_initrd
# @launchit.disable
if type(CONFIG).__name__ == 'BuildConfig' and HP.train.test_pack:
    coords = hp_parse_artifact_source(HP.train.test_pack)
    CONFIG.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', maven_group_id=lu.coalesce(coords.group_id, CONFIG.model_group_uri))
# @launchit.stop

## Create

In [107]:
# @launchit.disable_worker
assert HP.train.test_pack
coords = hp_parse_artifact_source(HP.train.test_pack)
test_pack_bytes = LS.artifact_registry.get_asset_content(coords.model_name, coords.model_version, asset_ext='pkl', maven_group_id=lu.coalesce(coords.group_id, CONFIG.model_group_uri))

with io.BytesIO(test_pack_bytes) as b:
    test_pack = pickle.load(b)

for key in test_pack:
    print(f'{key}.shape={test_pack[key].shape}')

pmasks.shape=torch.Size([5, 5])
obs.shape=torch.Size([5, 5, 1, 178, 152])
actions.shape=torch.Size([5, 5])
dones.shape=torch.Size([5, 5])


In [108]:
# @launchit.disable_worker
ump = hp_parse_universal_module(HP.train.optimizer)
assert not ump.args
optimizer = getattr(torch.optim, ump.module_name)(LS.world_model.parameters(), **ump.kwargs)

ump = hp_parse_universal_module(HP.train.learn_rate)
lr_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

ump = hp_parse_universal_module(HP.train.bce_loss_coef)
bce_loss_coef_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

ump = hp_parse_universal_module(HP.train.edge_loss_coef)
edge_loss_coef_anneal = get_anneal(ump.module_name, *ump.args, **ump.kwargs)

In [109]:
# @launchit.disable_worker
dataset = Dataset(
    envs_count=HP.rollout.envs_count,
    ob_shape=LS.world_model.params.ob_shape, 
    rollout_steps_count=HP.rollout.steps_count, 
    sequence_length=LS.world_model.params.sequence_length + 1,  # +1 is to satisfy demand for next obs in last item for WorldModel
    actions_count=LS.agent_params.actions_count,
    batches_count=(HP.rollout.envs_count * HP.rollout.steps_count) // HP.train.batch_size,
    is_shuffled=True,
)

for name, v in dataset.shared_tensors.items():
    if v is not None:
        assert v.is_shared(), f'{name} is not shared'
        LOG(f'{name:>28}: {str(v.device):>6}, {str(v.dtype):>15}, {v.shape}')

                      pmasks:    cpu,   torch.float32, torch.Size([1, 132, 5])
                         obs:    cpu,     torch.uint8, torch.Size([1, 132, 1, 178, 152])
                     actions:    cpu,     torch.int64, torch.Size([1, 132, 5])
                       dones:    cpu,   torch.float32, torch.Size([1, 132, 5])
                    env_inds:    cpu,     torch.int64, torch.Size([1, 132])
                   step_inds:    cpu,     torch.int64, torch.Size([1, 132])


In [110]:
# @launchit.disable_worker
edge_detector = EdgeDetector()
edge_detector = edge_detector.to(CONFIG.cuda_device)

## Train

In [111]:
# @launchit.disable_worker
metrics_suite = defaultdict(list)
timings = {}

# - reference images
# - images reporting

for epoch in tqdm(range(HP.train.epochs_count), disable=not CONFIG.is_interactive):
    timings.clear()
    t0 = time.time()
    
    # ANNEAL
    progress = epoch / HP.train.epochs_count
    lr = lr_anneal(progress)
    for param_group in optimizer.param_groups: param_group['lr'] = lr
    bce_loss_coef = bce_loss_coef_anneal(progress)
    edge_loss_coef = edge_loss_coef_anneal(progress)

    # ROLLOUT
    t1 = time.time()
    rollout_env_story = LS.rollout_env_story_sampler.sample()
    # Here agents could take over steering wheel from each other =)
    # Actually this is an interesting idea since agent could both help and hinder each other.
    # Also one could think towards mixture of experts
    agent_id = RNG.choice(list(LS.agents.keys()))
    agent = LS.agents[agent_id]
    LOG(f'Rollout: agent="{agent_id}", story="{rollout_env_story.desc}"', when=not CONFIG.is_interactive)
    rr = LS.rollout_manager.rollout(agent, LS.world_model, dataset, steps_count=HP.rollout.steps_count, ram=rollout_env_story.ram, ram_patch=rollout_env_story.ram_patch)
    LOG(f'Rollout done', when=not CONFIG.is_interactive)
    timings['rollout'] = time.time() - t1
    t1 = time.time()
    
    processed_batch_items = 0

    # OPTIMIZATION
    for batch in dataset:
        # Important batch items invariants and conditions are highlighted in "rollout: pmasks, obs, actions, dones"
        
        # Need at least one obs for prediction, so filter out batch items which don't have enough items
        batch_mask = batch.pmasks.sum(axis=1) < LS.world_model.params.sequence_length
        batch = batch._replace(
            obs=batch.obs[batch_mask],
            actions=batch.actions[batch_mask],
            pmasks=batch.pmasks[batch_mask],
            dones=batch.dones[batch_mask],
        )
        
        # AMP is req-d for FlashAttention
        with torch.amp.autocast(device_type=CONFIG.cuda_device, dtype=torch.bfloat16, enabled=HP.system.use_amp):
            assert batch.obs.shape[1] - 1 == LS.world_model.params.sequence_length
            batch_size = len(batch.obs)
            pmasks = batch.pmasks[:,:-1]

            # Feed world model with all sequence elements but the last one. Last element is for prediction
            raw_pred_next_obs = LS.world_model(obs=batch.obs[:,:-1], actions=batch.actions[:,:-1], padding_masks=pmasks)
            assert raw_pred_next_obs.shape == (batch_size, LS.world_model.params.sequence_length, *LS.world_model.ob_shape)
        
            raw_true_next_obs = batch.obs[:,1:]
            assert raw_true_next_obs.shape == (batch_size, LS.world_model.params.sequence_length, *LS.world_model.ob_shape)

            # Get only unmasked next_obs, unravel first 2 dims to get flat list of images
            flat_pmasks = pmasks.ravel()
            pred_next_obs = raw_pred_next_obs.view(-1, *LS.world_model.ob_shape) # (raw_uberbatch, *ob_shape)
            pred_next_obs = pred_next_obs[flat_pmasks] # (uberbatch, *ob_shape)
            true_next_obs = raw_true_next_obs.view(-1, *LS.world_model.ob_shape) # (raw_uberbatch, *ob_shape)
            true_next_obs = true_next_obs[flat_pmasks] # (uberbatch, *ob_shape)
            
            ####
            # 1. Pixel-level Loss (BCE scales cleanly for grayscale classification)
            bce_loss = F.binary_cross_entropy(pred_next_obs, true_next_obs)
            
            # 2. Edge-level Loss (Forces sharp silhouettes for small moving characters)
            pred_edges = edge_detector(pred_next_obs)
            true_edges = edge_detector(true_next_obs)
            edge_loss = F.mse_loss(pred_edges.magnitude, true_edges.magnitude)
            
            # Total composite loss
            loss = (bce_loss_coef * bce_loss) + (edge_loss_coef * edge_loss)
            
        optimizer.zero_grad()
        loss.backward()
        group_grad_norms = calc_group_grad_norms(LS.world_model)
        grad_norm = lu.when(
            HP.train.max_grad_norm is not None,
            lambda: torch.nn.utils.clip_grad_norm_(LS.world_model.parameters(), max_norm=HP.train.max_grad_norm),
            0
        )
        optimizer.step()

        processed_batch_items += batch_size

    LOG(f'Optimization done', when=not CONFIG.is_interactive)
    timings['optimization'] = time.time() - t1

    # REPORT
    LS.summary_writer.add_scalar('timings/a) eps', 1 / (time.time() - t0), epoch) # epochs per second, bigger = better
    LS.summary_writer.add_scalar('timings/b) rollout', timings['rollout'], epoch) # time per rollout, lesser = better
    LS.summary_writer.add_scalar('timings/c) optimization', timings['optimization'], epoch) # time per optimization, lesser = better
    
    LS.summary_writer.add_scalar('anneal/a) learn_rate', lr, epoch)
    LS.summary_writer.add_scalar('anneal/b) bce_loss_coef', bce_loss_coef, epoch)
    LS.summary_writer.add_scalar('anneal/c) edge_loss_coef', edge_loss_coef, epoch)
    
    LS.summary_writer.add_scalar('rollout/a) reward', rr['rollout_counters']['reward'], epoch)
    LS.summary_writer.add_scalar('rollout/b) true_resets', rr['rollout_counters']['fake_resets'], epoch)
    LS.summary_writer.add_scalar('rollout/c) fake_true', rr['rollout_counters']['true_resets'], epoch)

    LS.summary_writer.add_scalar('optimization/a) loss', loss.item(), epoch)
    LS.summary_writer.add_scalar('optimization/b) bce_loss', bce_loss.item(), epoch)
    LS.summary_writer.add_scalar('optimization/c) edge_loss', edge_loss.item(), epoch)
    LS.summary_writer.add_scalar('optimization/z) processed_batch_items', processed_batch_items, epoch)
    
    LS.summary_writer.add_scalar('grad/a) grad_norm', grad_norm.item(), epoch)
    
    for name, value in group_grad_norms.items(): 
        LS.summary_writer.add_scalar(f'b) grad/grad_norm_{name}', value.item(), epoch)
        
    LS.summary_writer.flush()

    LOG(f'Progress {100 * progress:.2f}% ({epoch:_} / {HP.train.epochs_count:_})', when=not CONFIG.is_interactive)

  0%|          | 0/4 [00:00<?, ?it/s]

Envs reset


AttributeError: 'WorldModel' object has no attribute 'sequence_length'

## Save

In [ ]:
assert False

In [ ]:
# @launchit.disable_worker
lc = HP.launch_component()

if lc.version != 0:
    with io.BytesIO() as b:
        torch.save(LS.agent.state_dict(), b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='pt', asset_classifier='agent', replace=True)
    
    with io.StringIO() as b:
        json.dump(dataclasses.asdict(LS.agent.params), b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='agent_params', replace=True)

    with io.StringIO() as b:
        json.dump(metrics_suite, b)
        LS.artifact_registry.attach_asset(lc.name, lc.version, b, asset_ext='json', asset_classifier='metrics_suite', replace=True)

    with open(CONFIG.metrics_suite_fname, 'w') as f:
        json.dump(metrics_suite, f)
        LOG(f'Metrics suite saved to "{CONFIG.metrics_suite_fname}"')

# LaunchIt!

## LAUNCH_NOTEBOOK

In [ ]:
# @launchit.disable
launchit_t0 = time.time()

In [64]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    LS.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=168
Creating /home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_16-launch168.ipynb
Created launch notebook "/home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_16-launch168.ipynb"


## DOCKER_LAUNCH_NOTEBOOK

In [50]:
# @launchit.disable
launchit_t0 = time.time()

In [51]:
# @launchit.collect_manual_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import launch_dispatcher
    image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
    launch_request = dict(
        launch_image=image_tag,
        keep_container=False,
    )
    launch_dispatcher.LaunchRequest.run(launch_request)
# @launchit.stop

In [52]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    lc = HP.launch_component()
    component_version = int(Autoincrement.get(lc.uri))
    assert component_version > 0, component_version
    LS.artifact_registry.register_component(lc.name, component_version)
    LOG(f'Model instance registered, version={component_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=component_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
    )
    launch_notebook_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=component_version, 
        expandvars=expandvars, 
        collect_inds=['initrd', 'build_docker_launch', 'manual_run_docker_launch', 'temp_config'], 
        disable_inds=[]
    )
    LOG(f'Created docker launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=1
Creating /home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_16-launch1.ipynb
Created docker launch notebook "/home/misha/dev/mine/neurolab/17_rl/17e_ppo_tr_atari_mp_16-launch1.ipynb"


## Optuna (model selection)

### Templates

In [15]:
# @launchit.collect_optuna
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK and os.path.exists('${OPTUNA_STUDY_FNAME}'):
    import re
    import launchit
    optuna_study_fname = '${OPTUNA_STUDY_FNAME}'
    optuna_study_storage = JournalStorage(JournalFileBackend(optuna_study_fname))
    optuna_study_name = '${OPTUNA_STUDY_NAME}'
    assert optuna_study_name
    assert optuna_study_name != '$' + '{OPTUNA_STUDY_NAME}', optuna_study_name
    optuna_study = optuna.load_study(study_name=optuna_study_name, storage=optuna_study_storage)
    optuna_trial = optuna_study.ask()
    optuna_study_serial = optuna_study.user_attrs['STUDY_SERIAL']
    optuna_study_nb_fname = re.sub('.optuna$', '.ipynb', optuna_study_fname)

    source_code = launchit.extract_source_code(optuna_study_nb_fname)
    Logging.get()(f'Extracted source code for set_hyperparamters from "{optuna_study_nb_fname}"')
    module = lu.make_module('optuna_study_hyperparameters', source_code)
    HP = module.set_hyperparameters(Hyperparameters(), optuna_study, optuna_trial)
    Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')

    assert os.path.exists(CONFIG.initrd_path)

    with open(os.path.join(CONFIG.initrd_path, 'optuna_trial.json'), 'w') as f:
        optuna_trial_dict = dict(
            trial_number=optuna_trial.number,
            study_serial=optuna_study_serial,
            study_name=optuna_study_name,
        )
        json.dump(optuna_trial_dict, f)

    with open(os.path.join(CONFIG.initrd_path, 'hyperparameters.json'), 'w') as f:
        json.dump(HP._asdict(), f)
        
elif CONFIG.exec_mode == ExecMode.DOCKER_LAUNCH_NOTEBOOK:
    hyperparameters_fname = os.path.join(CONFIG.initrd_path, 'hyperparameters.json')
    
    with open(hyperparameters_fname, 'r') as f:
        HP = Hyperparameters.from_dict(json.load(f))
        Logging.get()(f'HP loaded from "{hyperparameters_fname}"')
        Logging.get()(f'HP=\n{pprint.pformat(HP._asdict(), sort_dicts=False)}\n')


### optuna_run_docker_launch

In [16]:
# @launchit.collect_optuna_run_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    # optuna_study and optuna_trial are created in cell above
    assert optuna_study is not None
    assert optuna_trial is not None

    import launch_dispatcher
    short_image_tag = '${MODEL_NAME}' + ':' + '${MODEL_VERSION}'
    image_tag = os.path.join(CONFIG.docker_registry, short_image_tag)
    launch_request = dict(
        launch_image=image_tag,
        result_fname=os.path.join(project_root_path, CONFIG.relative_metrics_suite_fname),
        keep_container=False,
    )
    launch_result_metadata, launch_result_body = launch_dispatcher.LaunchRequest.run(launch_request)
    
    with open(CONFIG.self_fname + '.out', mode='wt') as out_file:
        if launch_result_metadata['is_ok']:
            if launch_result_body:
                decisive_metric = 'game_stats/video/reward'
                
                with io.BytesIO(launch_result_body) as b:
                    launch_result_dict = json.load(b)
    
                if launch_result_dict.get(decisive_metric, []):
                    scalar_result = np.array(launch_result_dict[decisive_metric]).mean() # reduce to just scalar e.g. via mean, sum, last
                    optuna_study.tell(optuna_trial, scalar_result, state=optuna.trial.TrialState.COMPLETE)
    
                    message = f'Trial {optuna_trial.number} ({short_image_tag}) finished with value: {scalar_result} and parameters: {optuna_trial.params}'
                    
                    try:
                        best_trial = optuna_study._get_best_trial(deepcopy=False)
                        message += f'. Best is trial {best_trial.number} with value: {best_trial.value}'
                    except ValueError:
                        # If no feasible trials are completed yet, study.best_trial raises ValueError
                        pass

                    out_file.write(message)
                else:
                    optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                    out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty/missing "{decisive_metric}"')
            else:
                optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
                out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) finished with empty result body: {launch_result_metadata=}')
        else:
            optuna_study.tell(optuna_trial, state=optuna.trial.TrialState.FAIL)
            out_file.write(f'Trial {optuna_trial.number} ({short_image_tag}) failed: {launch_result_metadata=}')

    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, message="To exit: use 'exit'")
    sys.exit(0)

### Unleash

In [61]:
# @launchit.disable
import launch_dispatcher
import concurrent.futures as cf
import subprocess

def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.self_name}'))
    assert model_version > 0, model_version
    LS.artifact_registry.register_component(CONFIG.self_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.self_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN.value,
        OPTUNA_STUDY_FNAME=optuna_study_fname,
        OPTUNA_STUDY_NAME=optuna_study_name,
    )
    launch_fname = launchit.launchit(
        CONFIG.self_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.self_name}:{model_version}', launch_fname

# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of docker a launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

optuna_study_name = '17e_study_23.3c'
optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
LOG(f'{optuna_study_name=}, {optuna_study_serial=}')
optuna_study_fname = os.path.join(CONFIG.subproject_path, 'optuna', optuna_study_name, optuna_study_name + '.optuna')
grid_search_space = None
optuna_study = optuna.create_study(
    study_name=optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True,
    sampler=lu.when(grid_search_space, lambda: optuna.samplers.GridSampler(grid_search_space), None),
)
optuna_study.set_user_attr('STUDY_SERIAL', optuna_study_serial)
launches_count = 6 * 4
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

optuna_study_name='17e_study_23.3c', optuna_study_serial='23.3c'


[I 2026-07-29 22:22:12,299] A new study created in Journal with name: 17e_study_23.3c


Model instance registered, version=144
Launching "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_16-launch144.ipynb"
6.0 idle runners exist, submitted launch "17e_ppo_tr_atari_mp_16:144"; running launches=1
Model instance registered, version=145
Launching "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_16-launch145.ipynb"
5.3 idle runners exist, submitted launch "17e_ppo_tr_atari_mp_16:145"; running launches=2
Model instance registered, version=146
Launching "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_16-launch146.ipynb"
4.3 idle runners exist, submitted launch "17e_ppo_tr_atari_mp_16:146"; running launches=3
Model instance registered, version=147
Launching "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_16-launch147.ipynb"
3.7 idle runners exist, submitted launch "17e_ppo_tr_atari_mp_16:147"; running launches=4
Model instance registered, version=148
Launching "/home/misha/dev/mine/neurolab/run/17_rl/17e_ppo_tr_atari_mp_16-launch1

In [ ]:
# @launchit.disable
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

# Docker build

In [71]:
# @launchit.collect_build_docker_launch
# @launchit.disable
if CONFIG.exec_mode == ExecMode.LAUNCH_NOTEBOOK:
    import subprocess
    import tempfile

    assert os.path.exists(os.path.join(build_project_root_path, CONFIG.relative_self_fname))

    dockerfile_content = f'''
FROM {os.path.join(CONFIG.docker_registry, 'neurolab_source:latest')}
USER 0
RUN pip install --break-system-packages --no-cache-dir ale_py==0.12.0+neurolab2 --index-url=http://nexus:8081/repository/neurolab-pypi/simple --trusted-host=nexus
USER 1000
WORKDIR /neurolab
RUN git pull --rebase
WORKDIR /neurolab/{CONFIG.subproject_name}
COPY --chown=1000:1000 {CONFIG.relative_self_fname} .
'''
    if os.path.exists(CONFIG.initrd_path):
        dockerfile_content += f'''
RUN mkdir -p /neurolab/{CONFIG.relative_initrd_path}
COPY --chown=1000:1000  {CONFIG.relative_initrd_path} /neurolab/{CONFIG.relative_initrd_path}
'''

    dockerfile_content += f'''
RUN touch /neurolab/.docker_launch
CMD ["papermill", "{os.path.basename(CONFIG.self_fname)}", "{os.path.basename(CONFIG.self_fname)}"]
    '''
    with tempfile.NamedTemporaryFile(mode='wt', suffix='.Dockerfile') as dockerfile:
        with open(dockerfile.name, 'w') as f:
            f.write(dockerfile_content)

        image_tag = os.path.join(CONFIG.docker_registry, '${MODEL_NAME}' + ':' + '${MODEL_VERSION}')
        subprocess.run(
            # --add-host is used to overcome problems with VPN (when VPN enabled build process cannot resolve nexus host)
            ['docker', 'buildx', 'build', '-f', dockerfile.name, '-t', image_tag, build_project_root_path, '--load', '--no-cache', '--network=host', '--add-host', 'nexus=127.0.0.1'],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
        subprocess.run(
            ['docker', 'push', image_tag],
            text=True,                # Handles input/output as strings instead of bytes
            capture_output=False,     # Streams Docker's build output directly to your terminal
            check=True                # Raises an exception if the command fails
        )
# @launchit.stop